# From two outputs to a decision: seven uncertainty summaries

This notebook walks through the saved Stage 4 study. It compares ways to summarize a predictive entropy map, shows how the rule was selected on validation, and reports what happened on held-out BUSI and external cohorts. The aggregate sections use files bundled in the repository. The [study guide](../stage_4_selective_prediction/SELECTIVE_PREDICTION_STUDY.md) links the method, protocol and result tables.

**Question:** Given a predicted segmentation mask and a predictive uncertainty map for each image, which image-level summary best ranks segmentation quality?

This study has already been run. The notebook visualizes saved results rather than retraining the model or selecting a new method.

1. MedCLIPSeg produces a **predicted mask** and a **predictive-entropy map** for each image.
2. Seven methods turn that same map into seven image-level scores. Reference annotations are used only afterward to evaluate Dice similarity coefficient (DSC) and normalized surface Dice (NSD).
3. On 65 lesion-positive BUSI validation images, compare score–quality association and risk–coverage performance (AURC), then select the primary method.
4. Freeze the selection and threshold; evaluate BUSI test, four external cohorts, and benign/malignant subgroups.
5. Handle empty masks separately and inspect failure modes.

**Why this order?** Select against a prespecified main objective on one comparable validation set, then examine where the selected method works or fails. Picking a new winner after looking at each test subgroup would bias evaluation. The ten scenarios organize limitations and future subgroup analyses; they do not replace validation-based selection.

Sources: [experimental protocol](../stage_4_selective_prediction/study_reference/EXPERIMENTAL_PROTOCOL.md), [theory and metrics](../stage_4_selective_prediction/study_reference/THEORY_AND_METRICS.md), [decisions and limitations](../stage_4_selective_prediction/study_reference/RESEARCH_DECISIONS_AND_LIMITATIONS.md).

The aggregate tables and plots run from the checkout. Set `MEDCLIPSEG_OUTPUT` to an external experiment output directory to enable the optional case-level sections.

In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# Set MEDCLIPSEG_REPO when launching outside the checkout, including in Colab.
cwd = Path(os.environ.get("MEDCLIPSEG_REPO", Path.cwd())).expanduser().resolve()
repo = next((p for p in (cwd, *cwd.parents) if (p / "research/stage_4_selective_prediction/experiments").is_dir()), None)
if repo is None:
    raise FileNotFoundError("Open the checkout or set MEDCLIPSEG_REPO to its root.")
experiments = repo / "research/stage_4_selective_prediction/experiments"
case_out = Path(os.environ.get("MEDCLIPSEG_OUTPUT", repo.parent / "data/outputs")).expanduser()
val = experiments / "validation_selection/results"
test = experiments / "locked_busi_test/results"
external_results = experiments / "external_test/results"
case_val = case_out / "BUSI/validation/seed666"
case_test_scores = case_out / "BUSI/aggregation_results/seed666/uncertainty_scores_BUSI_test_seed666.csv"
val_scores = case_val / "aggregation_results/uncertainty_scores_BUSI_val_seed666.csv"
methods = ['U_mean', 'U_positive', 'U_fg', 'U_boundary', 'U_threshold', 'U_std_fg', 'U_ratio']
print("Bundled experiment results:", experiments)
print("Optional case-level artifacts:", case_out)


### The experimental route

Every arrow below corresponds to a saved artifact or a comparison performed later in this notebook. The **ground-truth mask enters only at evaluation**. A score may be useful for ranking segmentation quality without diagnosing whether a lesion is benign or malignant.

In [ ]:
fig, ax = plt.subplots(figsize=(15, 2.8))
labels = ['Ultrasound image', 'Predicted mask\n+ entropy map', 'Seven image\nlevel scores', 'DSC / NSD\nassociation', 'Risk–coverage\ncomparison', 'Frozen rule\n+ subgroup audit']
for j, label in enumerate(labels):
    ax.text(j, 0.5, label, ha='center', va='center', fontsize=10,
            bbox=dict(boxstyle='round,pad=0.65', fc='#e9f3fa' if j < 3 else '#e7f4e9', ec='#46758e'))
    if j < len(labels)-1:
        ax.annotate('', xy=(j+0.66,0.5), xytext=(j+0.34,0.5), arrowprops=dict(arrowstyle='->', lw=1.8))
ax.set(xlim=(-0.6,5.6),ylim=(0,1)); ax.axis('off'); plt.show()

## 1. Research basis: why these seven summaries?

MedCLIPSeg uses probabilistic inference to produce a **predictive-entropy map**. Its appearance can differ from uncertainty maps made with deterministic models, so the experiment tests aggregation rules on *this model's output*. A bright contour indicates predictive uncertainty, not necessarily a contour error.

| Method | Summary | Motivation and relationship to prior work |
|---|---|---|
| U_mean | Mean over the entire map | Global AVG operation studied by Guarino et al.; a simple reference that may dilute small lesions. |
| U_positive | Mean where entropy > 0 | Our ablation for excluding exact zeros. |
| U_fg | Mean inside the **predicted mask** | Related to the per-structure mean in Roy et al.; does not reproduce the full QuickNAT method. |
| U_boundary | Mean in a morphological band around the mask | A contour-quality hypothesis motivated by spatial-uncertainty research; our own implementation. |
| U_threshold | Mean where entropy > 0.1 | Inspired by threshold aggregation in Guarino et al.; its empty-set rule differs. |
| U_std_fg | Spatial standard deviation inside the mask | Our ablation asking whether heterogeneity adds information beyond the mean. |
| U_ratio | U_fg / (U_mean + 10⁻⁶) | Our local/global contrast ablation; **not** Guarino's QFR. |

Koleilat et al. supply the model and its outputs; Roy provides the closest antecedent for U_fg; Guarino motivates comparing aggregation strategies. Jungo and Mehrtash motivate testing how spatial uncertainty relates to errors, without proposing these exact seven formulas. Geifman and El-Yaniv motivate risk–coverage analysis. DeVries and Taylor study a related, trained quality-estimation system. Our rule is applied after inference without additional training.

**Testable hypothesis:** if background uncertainty is low, U_mean combines lesion uncertainty with lesion area fraction. U_fg may rank segmentation quality better. This motivates the experiment; it does not guarantee superiority.

Papers: [MedCLIPSeg](https://openaccess.thecvf.com/content/CVPR2026/html/Koleilat_MedCLIPSeg_Probabilistic_Vision-Language_Adaptation_for_Data-Efficient_and_Generalizable_Medical_Image_CVPR_2026_paper.html) · [Roy et al.](https://doi.org/10.1016/j.neuroimage.2019.03.042) · [Guarino et al.](https://openaccess.thecvf.com/content/CVPR2026/html/Guarino_Better_than_Average_Spatially-Aware_Aggregation_of_Segmentation_Uncertainty_Improves_Downstream_CVPR_2026_paper.html) · [Jungo et al.](https://doi.org/10.3389/fnins.2020.00282) · [Mehrtash et al.](https://doi.org/10.1109/TMI.2020.3006437) · [Geifman and El-Yaniv](https://arxiv.org/abs/1705.08500) · [DeVries and Taylor](https://arxiv.org/abs/1807.00502).

### Paper-to-method map

This matrix separates **exact operation**, **conceptual motivation**, and **our own ablation**. A paper's name beside a method does not mean that the complete published algorithm was reproduced. Different models can also produce different uncertainty maps; the seven numbers here all come from MedCLIPSeg's predictive-entropy map.

In [ ]:
papers = ['Koleilat: source map','Roy: structure mean','Guarino: aggregation','Jungo / Mehrtash: error question','Geifman: selective risk']
relation = pd.DataFrame('—',index=methods,columns=papers)
relation.loc[:,papers[0]]='map'
relation.loc['U_fg',papers[1]]='operation'
relation.loc[['U_mean','U_threshold'],papers[2]]=['operation','inspired']
relation.loc[['U_positive','U_boundary','U_std_fg','U_ratio'],papers[2]]='ablation'
relation.loc[:,papers[3]]='motivation'
relation.loc[:,papers[4]]='evaluation'
display(relation.style.set_caption('How the literature shapes the seven-method experiment'))

## 2. One real case: mask, map, and seven scores

The mask and map below are saved files from the **same validation image**. The CSV stores the seven scores computed by the study's implementation. DSC and NSD use the reference annotation only for evaluation; they do not enter the seven scores.

In [ ]:
from PIL import Image
case = "120_benign.png"
case_mask = case_val / "seg_results/MedCLIPSeg_unimedclip_ViT-B-16_Prompt-original" / case
case_entropy = case_val / "unc_results/MedCLIPSeg_unimedclip_ViT-B-16_Prompt-original" / (Path(case).stem + ".npy")
has_example = all(p.is_file() for p in (val_scores, case_mask, case_entropy))
if has_example:
    seg = np.asarray(Image.open(case_mask))
    unc = np.load(case_entropy, allow_pickle=False)
    fig, ax = plt.subplots(1, 2, figsize=(9, 4))
    ax[0].imshow(seg, cmap='gray', vmin=0, vmax=255); ax[0].set_title('Predicted mask')
    im = ax[1].imshow(unc, cmap='magma'); ax[1].set_title('Uncertainty map')
    for a in ax: a.axis('off')
    fig.colorbar(im, ax=ax[1], fraction=.045, label='Entropy')
    plt.show()
    row = pd.read_csv(val_scores).set_index("image_id").loc[case]
    display(pd.DataFrame({'Method': methods, 'Score': [row[m] for m in methods]}).style.format({'Score': '{:.5f}'}))
    print(f'DSC={row.DSC:.4f}; NSD={row.NSD:.4f}; predicted-mask pixels={int(row.count_fg)}')
else:
    print("Optional validation mask, entropy or scores are unavailable.")


### What each aggregation actually reads

The shaded panels show **pixel selections**, not seven new uncertainty maps. All seven methods use the same entropy array. The foreground and boundary are based on the **predicted** mask; the threshold method uses entropy intensity instead. The table's scores are the official saved values, so the illustration below does not silently replace the implementation.

In [ ]:
if has_example:
    from scipy import ndimage
    fg = seg > 127
    band = ndimage.binary_dilation(fg, structure=np.ones((5,5))) ^ ndimage.binary_erosion(fg, structure=np.ones((5,5)))
    regions = {
        'U_mean': np.ones_like(fg, dtype=bool),
        'U_positive': unc > 0,
        'U_fg': fg,
        'U_boundary': band,
        'U_threshold': unc > 0.1,
        'U_std_fg': fg,
        'U_ratio': fg
    }
    fig, axes = plt.subplots(2,4,figsize=(14,6))
    for ax,m in zip(axes.flat,methods):
        ax.imshow(unc,cmap='magma',vmin=0,vmax=max(0.45,float(unc.max())))
        ax.contour(regions[m],levels=[.5],colors='cyan',linewidths=.65)
        ax.set_title(f'{m}: {row[m]:.4f}',fontsize=10); ax.axis('off')
    axes.flat[-1].axis('off')
    plt.suptitle('One entropy map; seven summaries (cyan outlines show selected pixels)',y=1.01)
    plt.tight_layout(); plt.show()
else:
    print("The pixel-region comparison requires the optional validation example.")


For a predicted-empty mask, U_fg, U_boundary, U_std_fg, and U_ratio are undefined. U_threshold can also be undefined if no pixel exceeds 0.1. Undefined is not the same as zero uncertainty.

## 3. Fair comparison on BUSI validation

All seven methods are compared on the **same 65 lesion-positive images**. The 13 normal images with empty predicted masks are excluded because several scores are undefined there. DSC measures region overlap; NSD measures surface agreement at the evaluator's tolerance. If higher uncertainty indicates poorer segmentation, Spearman correlation with DSC/NSD should be **negative**. Correlation describes ranking, not calibration or causation. AURC summarizes loss `1−DSC` (or `1−NSD`) among retained images across coverage levels: **lower is better**.

In [ ]:
rho = pd.read_csv(val / 'spearman_validation_65_lesion.csv').set_index('Metric')
aurc = pd.read_csv(val / 'validation_method_comparison_N65.csv').set_index('Method')
area = pd.read_csv(val / 'lesion_area_correlation.csv').set_index('Metric')
comparison = pd.DataFrame({
    'DSC rho': rho.loc[methods, 'DSC_rho'], 'NSD rho': rho.loc[methods, 'NSD_rho'],
    'DSC AURC ↓': aurc.loc[methods, 'AURC_DSC'], 'NSD AURC ↓': aurc.loc[methods, 'AURC_NSD'],
    'Predicted-area rho': area.loc[methods, 'Spearman_rho_Area']
}).sort_values('DSC AURC ↓')
display(comparison.style.format('{:.3f}'))
fig, ax = plt.subplots(figsize=(8, 4))
comparison['DSC AURC ↓'].sort_values().plot.barh(ax=ax, color='#377da8')
ax.invert_yaxis(); ax.set(xlabel='AURC for 1−DSC (lower is better)', ylabel='')
plt.tight_layout(); plt.show()

### Does higher uncertainty accompany lower Dice?

Each point below is a **real BUSI validation lesion image**. The same 65 cases appear in every panel. A downward pattern supports the intended interpretation: higher score, lower Dice. Compare the direction and strength of Spearman ρ; a score's numeric scale differs by method, so absolute x-values should not be compared across panels.

In [ ]:
if val_scores.is_file():
    scores = pd.read_csv(val_scores)
    lesions = scores.loc[scores['count_fg'] > 0].copy()
    fig, axes = plt.subplots(2,4,figsize=(16,7),sharey=True)
    for ax,m in zip(axes.flat,methods):
        ax.scatter(lesions[m],lesions['DSC'],s=22,alpha=.65,c='#356b9b')
        ax.set_title(f'{m}  |  ρ={rho.loc[m,"DSC_rho"]:+.2f}')
        ax.set(xlabel='Uncertainty score',ylabel='Dice (DSC)')
        ax.grid(alpha=.2)
    axes.flat[-1].axis('off')
    plt.suptitle('BUSI validation: uncertainty score versus segmentation quality (n=65)',y=1.01)
    plt.tight_layout(); plt.show()
else:
    print("Validation scatter plots require the optional per-image scores.")


### A second question: can the score prioritize review?

Correlation alone is not enough. A selective workflow retains low-score images and asks for review of high-score images. **Risk** is the mean segmentation loss among retained images; **coverage** is the retained fraction. The saved risk–coverage curves and AURC table below answer whether the ordering helps across many retention levels. Oracle uses ground truth and is an unattainable reference; random retention is an uninformed control.

In [ ]:
from IPython.display import Image as NotebookImage
curve_path = val / 'figure1_risk_coverage_curves.png'
display(NotebookImage(filename=str(curve_path), width=950))
bench = aurc.loc[methods, ['AURC_DSC','AURC_NSD']].copy().sort_values('AURC_DSC')
display(bench.style.format('{:.4f}').set_caption('Saved validation AURC; smaller is better'))

### Why did a global average underperform?

With foreground fraction `α`, the map-wide mean satisfies **U_mean = α·U_fg + (1−α)·U_bg** exactly. If background entropy is small, lesion size can dominate the global score. This suggests a mechanism, but the data still have to test it. The next plot uses the saved per-image predicted area and uncertainty scores. Spearman ρ quantifies the observed monotonic association; it does not establish causality.

In [ ]:
if val_scores.is_file():
    fig,axes=plt.subplots(1,2,figsize=(11,4))
    for ax,m in zip(axes,['U_mean','U_fg']):
        ax.scatter(lesions['count_fg']/lesions['count_total'],lesions[m],c=lesions['DSC'],cmap='viridis',s=33,alpha=.8)
        ax.set(xlabel='Predicted foreground fraction',ylabel=m,
               title=f'{m} vs area: ρ={area.loc[m,"Spearman_rho_Area"]:+.3f}')
        ax.grid(alpha=.2)
    plt.suptitle('BUSI validation: color represents actual Dice quality')
    plt.tight_layout(); plt.show()
else:
    print("The area comparison requires the optional per-image scores.")


**Reading the result:** U_fg has the lowest observed validation DSC AURC (0.0839), but U_std_fg is close (0.0850). Their paired difference was unresolved (`p=0.294`). U_fg was fixed as the primary method because it is simple and has the lowest point estimate; U_std_fg remains a comparator. U_mean was strongly associated with predicted area (`ρ≈+0.915`), compared with `ρ≈−0.085` for U_fg. Near-zero area correlation does not prove independence.

The best choice depends on the **task**: for DSC/NSD quality ranking on this validation cohort, U_fg leads narrowly; for an empty predicted mask, U_fg is undefined. No universal winner is established.

### Is the small difference between the leading two resolved?

A lower point estimate is not enough to claim that U_fg beats U_std_fg. The paired bootstrap compares both methods on the **same images**. A 95% interval that crosses zero leaves the difference unresolved.

In [ ]:
paired = pd.read_csv(val / 'paired_bootstrap_aurc_ci.csv')
show = paired.loc[paired['Metric_A'].eq('U_fg'), ['Metric_B','Delta_AURC_DSC','CI95_DSC_Lower','CI95_DSC_Upper','p_val_DSC']].copy()
display(show.style.format({'Delta_AURC_DSC':'{:+.4f}','CI95_DSC_Lower':'{:+.4f}','CI95_DSC_Upper':'{:+.4f}','p_val_DSC':'{:.3f}'}))
fig,ax=plt.subplots(figsize=(8,3))
for j,r in enumerate(show.itertuples()):
    ax.plot([r.CI95_DSC_Lower,r.CI95_DSC_Upper],[j,j],color='#27789c',lw=3)
    ax.plot(r.Delta_AURC_DSC,j,'o',color='#123f59')
ax.axvline(0,color='crimson',ls='--',lw=1)
ax.set_yticks(range(len(show)),show['Metric_B']);ax.set(xlabel='Paired ΔAURC: U_fg minus comparator',title='BUSI validation: 95% bootstrap intervals')
ax.invert_yaxis();ax.grid(axis='x',alpha=.2);plt.tight_layout();plt.show()

## 4. Frozen selection and external evaluation

The cutoff `τ_fg=0.064058324` was set on BUSI validation to target 80% retention among lesion-positive cases. It was then used unchanged on BUSI test and four external cohorts. The table below **reads the saved CSVs** and shows all seven methods in each cohort. A cohort's point-estimate winner may change; this does not reselect the primary policy.

In [ ]:
cohorts = ['BUSI', 'UDIAT', 'BUID', 'BUSUC', 'BUSBRA']
records = []
for name in cohorts:
    cohort_results = external_results / ('BUSI_reference' if name == 'BUSI' else name)
    t = pd.read_csv(cohort_results / f'method_comparison_{name}.csv').set_index('Method')
    for m in methods:
        records.append({'Cohort': name, 'Method': m, 'DSC AURC': t.loc[m, 'AURC_DSC'], 'NSD AURC': t.loc[m, 'AURC_NSD']})
external = pd.DataFrame(records)
pivot = external.pivot(index='Method', columns='Cohort', values='DSC AURC').reindex(methods)
display(pivot.style.format('{:.3f}').highlight_min(axis=0, color='#d8f1df'))
winners = external.loc[external.groupby('Cohort')['DSC AURC'].idxmin(), ['Cohort', 'Method', 'DSC AURC']].reset_index(drop=True)
print('Lowest DSC AURC by test cohort:')
display(winners)
print(f"U_fg had the lowest DSC AURC in {(winners['Method'] == 'U_fg').sum()} of {len(winners)} test cohorts.")
op = pd.read_csv(test / 'primary_operating_point_test.csv').iloc[0]
print(f"BUSI test: coverage {op.Realized_Test_Coverage:.1%}; baseline DSC {op.Baseline_DSC:.4f}; retained DSC {op.Accepted_DSC:.4f}")

### Across cohorts: ranking, not a new selection

The following graphic displays all seven DSC AURCs by test cohort. Lower is better within each column. U_fg has the lowest point estimate in four of five test cohorts; U_std_fg is slightly lower in BUSUC. These are five observed samples, not a statistical claim that U_fg will win 80% of future settings.

In [ ]:
fig,ax=plt.subplots(figsize=(11,5))
mat=pivot.to_numpy()
im=ax.imshow(mat,aspect='auto',cmap='YlGnBu_r')
ax.set_xticks(range(len(pivot.columns)),pivot.columns)
ax.set_yticks(range(len(pivot.index)),pivot.index)
for y in range(mat.shape[0]):
    for x in range(mat.shape[1]):
        ax.text(x,y,f'{mat[y,x]:.3f}',ha='center',va='center',fontsize=9)
fig.colorbar(im,ax=ax,label='DSC AURC (lower is better)')
ax.set_title('Frozen test cohorts: all seven methods on matched cases')
plt.tight_layout();plt.show()

### A frozen cutoff changes coverage across domains

The same score threshold is used everywhere. Coverage can fall under distribution shift; higher retained DSC then applies to a smaller subset. Plot both dimensions before calling a deployment result successful.

In [ ]:
import json
rows=[]
for name in cohorts:
    if name=='BUSI':
        pnt=pd.read_csv(test/'primary_operating_point_test.csv').iloc[0]
    else:
        pnt=pd.read_csv(external_results / name/f'primary_operating_point_{name}.csv').iloc[0]
    if name == 'BUSI':
        coverage, retained, baseline = pnt['Realized_Test_Coverage'], pnt['Accepted_DSC'], pnt['Baseline_DSC']
    else:
        coverage, retained = pnt['realized_coverage'], pnt['accepted_dsc']
        baseline = json.loads((external_results / name / f'summary_evaluation_{name}.json').read_text())['baseline_stats']['mean_dsc']
    rows.append({'Cohort': name, 'Coverage': float(coverage),
                 'Baseline DSC': float(baseline), 'Retained DSC': float(retained)})
transfer=pd.DataFrame(rows)
display(transfer.style.format({'Coverage':'{:.1%}','Baseline DSC':'{:.3f}','Retained DSC':'{:.3f}'}))
fig,axes=plt.subplots(1,2,figsize=(11,3.8))
axes[0].bar(transfer.Cohort,transfer.Coverage,color='#4a8fa8');axes[0].axhline(.8,color='crimson',ls='--',label='Validation target')
axes[0].set(ylabel='Retained fraction',ylim=(0,1),title='Coverage with frozen cutoff');axes[0].legend()
x=np.arange(len(transfer));w=.35
axes[1].bar(x-w/2,transfer['Baseline DSC'],w,label='All cases',color='#94a9b8')
axes[1].bar(x+w/2,transfer['Retained DSC'],w,label='Retained subset',color='#3b9271')
axes[1].set_xticks(x,transfer.Cohort);axes[1].set(ylim=(0,1),ylabel='Mean DSC',title='Quality and selection');axes[1].legend()
plt.tight_layout();plt.show()

**Interpretation:** AURC measures how well a score ranks cases; it does not improve the original masks. Mean DSC rises among retained cases because some images are excluded, reducing coverage. External evaluation tests transfer of the frozen rule, not the absence of domain shift. Pooled external AURC is a separate calculation, not the average of the four cohort AURCs.

## 5. Benign, malignant, and normal images

Benign/malignant are **dataset labels**, not diagnoses made by the seven scores. We compare coverage and DSC under the frozen cutoff. Normal BUSI images often have an empty predicted mask and use the separate U_mean branch. Mixing them into the lesion-only comparison would change the statistical question.

In [ ]:
sub = pd.read_csv(test / 'subgroup_pathology_breakdown_test.csv')
display(sub[['Subgroup', 'Sample_Count', 'Realized_Coverage', 'Baseline_DSC', 'Accepted_DSC', 'Rejected_DSC']].style.format({'Realized_Coverage': '{:.1%}', 'Baseline_DSC': '{:.3f}', 'Accepted_DSC': '{:.3f}', 'Rejected_DSC': '{:.3f}'}))
print('BUSI test: 13 normal images had both an empty prediction and an empty annotation. They do not test detection of a missed lesion.')

### Do the seven scores behave similarly within pathology labels?

The saved test cohort includes 44 benign and 21 malignant lesion images. The plot below shows **descriptive** within-group Spearman associations for each score against DSC. These smaller groups cannot establish a new subgroup-specific winner, and benign/malignant labels are not predicted by uncertainty. They do reveal whether a method's score–quality relationship looks different across the two groups.

In [ ]:
if case_test_scores.is_file():
    test_scores=pd.read_csv(case_test_scores)
    test_lesions=test_scores[test_scores.count_fg>0].copy()
    test_lesions['Pathology']=test_lesions.image_id.str.extract(r'_(benign|malignant)\.png$',expand=False)
    from scipy.stats import spearmanr
    subcorr=[]
    for group,g in test_lesions.groupby('Pathology'):
        for m in methods:
            subcorr.append({'Pathology':group,'Method':m,'rho DSC':spearmanr(g[m],g.DSC).statistic,'N':len(g)})
    subcorr=pd.DataFrame(subcorr)
    fig,ax=plt.subplots(figsize=(10,4))
    x=np.arange(len(methods));w=.36
    for shift,group,color in [(-w/2,'benign','#3a8f9d'),(w/2,'malignant','#be6b58')]:
        vals=subcorr[subcorr.Pathology.eq(group)].set_index('Method').loc[methods,'rho DSC']
        ax.bar(x+shift,vals,w,label=f'{group} (n={subcorr[subcorr.Pathology.eq(group)].N.iloc[0]})',color=color)
    ax.axhline(0,color='black',lw=.8);ax.set_xticks(x,methods,rotation=30,ha='right')
    ax.set(ylabel='Spearman ρ with DSC',title='BUSI test: descriptive score–quality association by pathology')
    ax.legend();ax.grid(axis='y',alpha=.2);plt.tight_layout();plt.show()
else:
    print("The subgroup correlations require the optional per-image BUSI test scores.")


### The empty-mask question emerges from the same experiment

Predicted-empty images expose a mathematical gap: a foreground mean has no region to average. The BUSI normal cases happen to be empty/empty, yet a missed lesion could also yield an empty prediction. The two-stage rule below is an **operational fallback**, not a benign/malignant classifier or a guarantee that an empty scan is truly normal.

In [ ]:
if case_test_scores.is_file():
    normal = test_scores[test_scores.count_fg.eq(0)]
    print('BUSI test empty predictions:', len(normal))
    print('Reference quality of observed empty/empty cases: DSC range', normal.DSC.min(), 'to', normal.DSC.max())
    print('U_fg defined:', normal.U_fg.notna().sum(), 'of', len(normal), '| U_mean defined:', normal.U_mean.notna().sum())
fig,ax=plt.subplots(figsize=(11,2.5));ax.axis('off')
for x,label,color in [(0.1,'Predicted mask\nnonempty','#e9f3fa'),(.48,'U_fg ≤ 0.064058324?\nretain / review','#e7f4e9'),(.1,'Predicted mask\nempty','#fff2df'),(.48,'U_mean ≤ 0.0001?\nretain / review','#fff2df')]:
    y=.7 if color!='#fff2df' else .25
    ax.text(x,y,label,transform=ax.transAxes,ha='center',va='center',bbox=dict(boxstyle='round,pad=.5',fc=color,ec='#666'))
ax.annotate('',xy=(.38,.7),xytext=(.23,.7),xycoords='axes fraction',arrowprops=dict(arrowstyle='->'))
ax.annotate('',xy=(.38,.25),xytext=(.23,.25),xycoords='axes fraction',arrowprops=dict(arrowstyle='->'))
ax.text(.8,.48,'Low uncertainty ≠\nno lesion',transform=ax.transAxes,ha='center',va='center',color='crimson',fontsize=13)
plt.show()

## 6. Ten scenarios: available evidence and next action

These scenarios are **review questions**, not ten validated clinical strata. The CSVs contain mask presence, benign/malignant/normal labels, quality, and uncertainty scores. They do not verify morphology such as spiculation, acoustic shadowing, density, or cyst type. Specific numerical values previously attached to these phenotypes in an architecture table are therefore not treated as empirical evidence here.

| Scenario | Available evidence | Appropriate action or study |
|---|---|---|
| 1. Well-defined benign lesion | BUSI benign subgroup measured; individual morphology unverified | Apply U_fg policy and inspect case-level DSC. |
| 2. Malignant lesion with difficult border | Malignant subgroup measured; spiculation unverified | Compare U_fg with quality and review accepted failures. |
| 3. Normal image, empty mask | 13 validation and 13 test BUSI empty predictions matched empty annotations | Use U_mean branch; do not infer absence of disease. |
| 4. Missed lesion, empty mask | Synthetic counterexample; no confirmed clinical instance in these records | Human review; the rule can accept a false negative. |
| 5. Small lesion | Predicted areas and documented `120_benign` case available | Stratify by area before claiming a small-lesion advantage. |
| 6. Acoustic shadow mistaken for a mass | Acoustic label unverified | Annotate the phenomenon and audit false positives. |
| 7. Multiple lesions | Multiplicity not systematically verified | Label component counts and evaluate that subgroup. |
| 8. Dense tissue and speckle | Density label unverified | Add a density label, then compare error and coverage. |
| 9. Atypical cyst | Subtype unverified | Obtain subtype labels and run a dedicated analysis. |
| 10. Domain shift | Four external cohorts and coverage change measured | Report cohort-level AURC and coverage without assigning a scanner cause. |

A full **7 methods × 10 phenotypes** comparison would require case-level phenotype labels. Current files cannot establish a winner for every scenario.

### Evidence strength across the ten scenarios

The matrix makes the distinction visible: **measured** means a field or subgroup is represented in saved data; **partial** means the broad mechanism is observed but the named phenotype is not verified; **unverified** means no case-level label supports that scenario. A scenario-specific seven-method winner requires labels and enough cases in that scenario.

In [ ]:
scenario_names=['Benign lesion','Malignant difficult border','Normal + empty mask','Missed lesion + empty mask','Small lesion','Acoustic shadow','Multiple lesions','Dense tissue / speckle','Atypical cyst','Domain shift']
status=['partial','partial','measured','synthetic only','partial','unverified','unverified','unverified','unverified','measured']
colors={'measured':'#4a9c75','partial':'#e6b655','synthetic only':'#e8866b','unverified':'#c7cbd0'}
fig,ax=plt.subplots(figsize=(11,5.5))
for j,(name,st) in enumerate(zip(scenario_names,status)):
    ax.barh(j,1,color=colors[st],height=.72)
    ax.text(.03,j,st.upper(),va='center',fontweight='bold',fontsize=9)
ax.set_yticks(range(10),scenario_names);ax.invert_yaxis();ax.set_xlim(0,1);ax.set_xticks([])
ax.set_title('What the saved data support about the ten proposed situations')
for spine in ax.spines.values():spine.set_visible(False)
plt.tight_layout();plt.show()

### A concrete accepted failure

The study also contains a direct counterexample to “low uncertainty means good segmentation.” The image below was **accepted** by the frozen U_fg threshold despite a very low DSC. This case should remain visible in any presentation of the method.

In [ ]:
bad_mask = case_out / "BUSI/seg_results/seed666/MedCLIPSeg_unimedclip_ViT-B-16_Prompt-original/116_malignant.png"
bad_entropy = case_out / "BUSI/unc_results/seed666/MedCLIPSeg_unimedclip_ViT-B-16_Prompt-original/116_malignant.npy"
if all(p.is_file() for p in (case_test_scores, bad_mask, bad_entropy)):
    bad=test_scores.loc[test_scores.image_id.eq('116_malignant.png')].iloc[0]
    bad_seg = np.asarray(Image.open(bad_mask))
    bad_map = np.load(bad_entropy, allow_pickle=False)
    fig,axes=plt.subplots(1,2,figsize=(8,3.6))
    axes[0].imshow(bad_seg,cmap='gray');axes[0].set_title('Predicted mask')
    im=axes[1].imshow(bad_map,cmap='magma');axes[1].set_title('Predictive entropy')
    for a in axes:a.axis('off')
    fig.colorbar(im,ax=axes[1],fraction=.045)
    plt.suptitle(f'Accepted failure: DSC={bad.DSC:.3f}, U_fg={bad.U_fg:.5f} ≤ 0.06406')
    plt.tight_layout();plt.show()
else:
    print("The failure example requires the optional BUSI test artifacts.")


## 7. Final rule and limitations

- **Nonempty predicted mask:** use U_fg; retain if `U_fg ≤ 0.064058324`, otherwise send for review.
- **Empty predicted mask:** U_fg is undefined; use U_mean with a separate `0.0001` cutoff. Low uncertainty does not confirm that no lesion exists.
- **Observed failure:** `116_malignant.png` on BUSI test had DSC≈0.2193 and was accepted by U_fg. The rule cannot guarantee detection of every segmentation error.

**Conclusion:** U_fg is the primary choice *for ranking lesion-segmentation quality under this protocol*, with U_std_fg statistically close. U_mean makes the empty-mask branch computable, but that branch does not establish safety against missed lesions. Benign/malignant and external-cohort analyses describe behavior; they do not select a new method or provide automated diagnosis.